# Leitura e processamento de dados de mudança de endereço
Neste notebook, os dados de registros do CadÚnico são processados para identificar mudanças

In [1]:
import pandas as pd
import geopandas as gpd
import h3

from datetime import datetime, timedelta
from pyarrow.parquet import ParquetFile
import pyarrow as pa

## 1. Parsing do arquivo

In [2]:
# Nomes das colunas
pf = ParquetFile('resultado_geocodificacao_19012026.parquet') 
first_rows = next(pf.iter_batches(batch_size = 1)) 
df = pa.Table.from_batches([first_rows]).to_pandas() 
i = list(first_rows.column_names)
print(i)

['CO_FAMILIAR_FAM', 'DT_ATUALIZACAO_FAM', 'VL_RENDA_MEDIA_FAM', 'NO_LOCALIDADE_FAM', 'NO_TIP_LOGRADOURO_FAM', 'NO_TIT_LOGRADOURO_FAM', 'NO_LOGRADOURO_FAM', 'NU_LOGRADOURO_FAM', 'DS_COMPLEMENTO_FAM', 'DS_COMPLEMENTO_ADIC_FAM', 'NU_CEP_LOGRADOURO_FAM', 'CO_ESPECIE_DOMIC_FAM', 'QT_COMODOS_DOMIC_FAM', 'QT_COMODOS_DORMITORIO_FAM', 'CO_MATERIAL_PISO_FAM', 'CO_MATERIAL_DOMIC_FAM', 'CO_AGUA_CANALIZADA_FAM', 'CO_ABASTE_AGUA_DOMIC_FAM', 'CO_BANHEIRO_DOMIC_FAM', 'CO_ESCOA_SANITARIO_DOMIC_FAM', 'CO_DESTINO_LIXO_DOMIC_FAM', 'CO_ILUMINACAO_DOMIC_FAM', 'IN_FAMILIA_INDIGENA_FAM', 'NO_RESERVA_INDIGENA_FAM', 'IN_FAMILIA_QUILOMBOLA_FAM', 'NO_COMUNIDADE_QUILOMBOLA_FAM', 'QT_FAMILIAS_DOMIC_FAM', 'VL_DESP_ENERGIA_FAM', 'VL_DESP_AGUA_ESGOTO_FAM', 'VL_DESP_GAS_FAM', 'VL_DESP_ALIMENTACAO_FAM', 'VL_DESP_TRANSPOR_FAM', 'VL_DESP_ALUGUEL_FAM', 'VL_DESP_MEDICAMENTOS_FAM', 'NU_ESTBO_SAUDE', 'IN_PARC_MDS_FAM', 'DATA_ARQUIVO', 'NOME_LOGRADOURO_FAM', 'ENDERECO_COMPLETO', 'NOME_TIT_LOGRADOURO_FAM', 'LATITUDE', 'LONGITUD

In [7]:
vars = ['CO_FAMILIAR_FAM', 
        'DT_ATUALIZACAO_FAM',
        'VL_RENDA_MEDIA_FAM',
        'CO_ESPECIE_DOMIC_FAM', 
        'QT_COMODOS_DOMIC_FAM', 
        'QT_COMODOS_DORMITORIO_FAM',
        'QT_FAMILIAS_DOMIC_FAM',
        'CO_BANHEIRO_DOMIC_FAM',
        'CO_DESTINO_LIXO_DOMIC_FAM',
        'CO_ILUMINACAO_DOMIC_FAM',
        'VL_DESP_ENERGIA_FAM', 
        'VL_DESP_AGUA_ESGOTO_FAM', 
        'VL_DESP_GAS_FAM', 
        'VL_DESP_ALIMENTACAO_FAM', 
        'VL_DESP_TRANSPOR_FAM', 
        'VL_DESP_ALUGUEL_FAM', 
        'VL_DESP_MEDICAMENTOS_FAM',
        'LATITUDE', 'LONGITUDE'
        ]

df =pd.read_parquet('../../../dados/_cofre/MDS_CadÚnico_2025/CadUnico/resultado_geocodificacao_19012026.parquet', columns=vars)

In [8]:
# Organização do dataframe
df = df.sort_values(by=['CO_FAMILIAR_FAM', 'DT_ATUALIZACAO_FAM'])
df = df.drop_duplicates(subset=['CO_FAMILIAR_FAM',
                                'DT_ATUALIZACAO_FAM',
                                'LATITUDE', 'LONGITUDE'
                                ], keep='last')

In [9]:
# Ajustes em colunas
for i in [
        'VL_DESP_ENERGIA_FAM', 
        'VL_DESP_AGUA_ESGOTO_FAM', 
        'VL_DESP_GAS_FAM', 
        'VL_DESP_ALIMENTACAO_FAM', 
        'VL_DESP_TRANSPOR_FAM', 
        'VL_DESP_ALUGUEL_FAM', 
        'VL_DESP_MEDICAMENTOS_FAM',
        ]:
    df[i] = df[i].fillna(0)
    df[i] = df[i].astype(int)

df['tt_despesas'] = df['VL_DESP_ENERGIA_FAM'] + df['VL_DESP_AGUA_ESGOTO_FAM'] + df['VL_DESP_GAS_FAM'] + df['VL_DESP_ALIMENTACAO_FAM'] + df['VL_DESP_TRANSPOR_FAM'] + df['VL_DESP_ALUGUEL_FAM'] + df['VL_DESP_MEDICAMENTOS_FAM']
df['pct_aluguel_despesas'] = round(df['VL_DESP_ALUGUEL_FAM']*100/df['tt_despesas'],2)
df['VL_RENDA_MEDIA_FAM'] = df['VL_RENDA_MEDIA_FAM'].astype(float)
df['DT_ATUALIZACAO_FAM'] = df['DT_ATUALIZACAO_FAM'].apply(lambda x: datetime.strptime(x, '%Y-%m-%d'))
df = df.sort_values(by=['CO_FAMILIAR_FAM', 'DT_ATUALIZACAO_FAM'])
df['num_registro'] = df.groupby('CO_FAMILIAR_FAM').cumcount()+1

## 2. Identificação das mudanças de endereço

In [11]:
#df_desloc = df.query('prox_num > num_registro')
datacols = vars + ['tt_despesas', 'pct_aluguel_despesas', 'num_registro']

df_desloc = df[datacols]
for col in datacols:
    df_desloc[f'{col}_pos'] = df_desloc[col].shift(-1)

df_desloc.query('CO_FAMILIAR_FAM == CO_FAMILIAR_FAM_pos & num_registro < num_registro_pos & LATITUDE != LATITUDE_pos & LONGITUDE != LONGITUDE_pos', inplace=True)
df_desloc = df_desloc.drop_duplicates(subset=['CO_FAMILIAR_FAM', 'DT_ATUALIZACAO_FAM'], keep='last')
df_desloc = df_desloc.drop(columns=['CO_FAMILIAR_FAM_pos', 'num_registro_pos'])
df_desloc['id_mudanca'] = df_desloc.index

In [12]:
def average_datetime(dates):
    reference = datetime(1900, 1, 1)
    return reference + sum([d - reference for d in dates], timedelta()) / len(dates)

df_desloc['data_media'] = df_desloc[['DT_ATUALIZACAO_FAM', 'DT_ATUALIZACAO_FAM_pos']].apply(lambda x: average_datetime(x), axis=1)

In [15]:
df_desloc = df_desloc.dropna(subset=['LATITUDE', 'LONGITUDE', 'LATITUDE_pos', 'LONGITUDE_pos'], how='any')

In [16]:
#df_desloc.to_csv('deslocamentos.csv', sep='\t', index=False)
df_desloc.to_parquet('deslocamentos.parquet', engine='pyarrow')